# Simulating a nonlinear system: Open-Loop/Closed-loop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro501/mass_spring_nonlinear_simulation.ipynb)

1. **Part 0** — first-order state-space form: $\dot{\mathbf{x}} = f(\mathbf{x}, u, t)$, $\mathbf{y} = h(\mathbf{x}, u, t)$, $u = \mathrm{ctl}(\mathbf{y}, r, t)$.
2. **Part 1** — one example, with NumPy, SciPy and Matplotlib.
3. **Part 2** — the same example, with minilink.

The same three steps, twice, with **NumPy / SciPy / Matplotlib** only, then with the **minilink** toolbox:

1. **Equations of motion in state-space form**: $\dot{\mathbf{x}} = f(\mathbf{x}, u, t)$, $\mathbf{y} = h(\mathbf{x}, u, t)$.
2. **Open-loop simulation**: an input signal $u(t)$ drives the plant.
3. **Closed-loop simulation**: a reference signal $r(t)$ and a control law $u = \mathrm{ctl}(\mathbf{y}, r, t)$.

Each part is written as a template: replace $f$ and $h$ with your own equations, $u(t)$ and $r(t)$ with your own signals, and $\mathrm{ctl}$ with your own control law.

## Part 0 — What every tool starts from

A simulator is at its core an ODE (ordinary differnetial equations) solver that integrates first-order equations. What it receives is a function of the state and of time,

$$
\dot{\mathbf{x}} = f_{ode}(\mathbf{x}, t).
$$

For system where we control an input $u$ and observe an output $y$, it is usefull to first form a more generic differential equation with the following form:
$$
\dot{\mathbf{x}} = f(\mathbf{x}, u, t), \qquad \mathbf{y} = h(\mathbf{x}, u, t).
$$

The same equations can be driven by a chosen signal, to see how the plant reacts to inputs, or by a control law, to see the closed loop. Either choice removes $u$ and returns the function the solver integrates:

**Open loop.** The input is a chosen signal $u(t)$:

$$f_{ol}(\mathbf{x}, t) = f(\mathbf{x}, u(t), t).$$

**Closed loop.** The input is a control law $\mathrm{ctl}\bigl(y,r,t)$ of the output, the reference signal $r(t)$, and time:

$$
f_{cl}(\mathbf{x}, t) = f(\mathbf{x}, u, t) 
= f(\mathbf{x}, \mathrm{ctl}\bigl(h(\mathbf{x}, u, t),\ r(t),\ t\bigr), t)
$$

A second-order equation, a higher-order equation, several coupled equations, or a transfer function is not yet the plant above. In each case the state $\mathbf{x}$ is chosen so that one vector equation gives $\dot{\mathbf{x}} = f(\mathbf{x}, u, t)$.

### Four Exemples

**1. Second order.** One mass, one equation:

$$m\ddot{q} + b\dot{q} + k q = u.$$

The state $\mathbf{x} = [q,\ \dot{q}]$ carries the position and the velocity, so the derivative is first order only:

$$
\dot{\mathbf{x}} = f(\mathbf{x}, u, t) = \begin{bmatrix} \dot{q} \\ (u - b\dot{q} - k q)/m \end{bmatrix}.
$$

**2. Third order.** The scalar equation

$$\dddot{q} + a\ddot{q} + b\dot{q} + c q = u$$

needs the acceleration in the state as well, $\mathbf{x} = [q,\ \dot{q},\ \ddot{q}]$:

$$
\dot{\mathbf{x}} = f(\mathbf{x}, u, t) = \begin{bmatrix} \dot{q} \\ \ddot{q} \\ u - a\ddot{q} - b\dot{q} - c q \end{bmatrix}.
$$

**3. Two coupled equations** Two masses linked by a spring,

$$m_1\ddot{q}_1 = -k(q_1 - q_2) + u, \qquad m_2\ddot{q}_2 = -k(q_2 - q_1),$$

become a single first-order equation on $\mathbf{x} = [q_1,\ q_2,\ \dot{q}_1,\ \dot{q}_2]$:

$$
\dot{\mathbf{x}} = f(\mathbf{x}, u, t) = \begin{bmatrix} \dot{q}_1 \\ \dot{q}_2 \\ \bigl(-k(q_1 - q_2) + u\bigr)/m_1 \\ -k(q_2 - q_1)/m_2 \end{bmatrix}.
$$

**4. A transfer function.** A strictly proper transfer function

$$\frac{Y(s)}{U(s)} = \frac{b_1 s + b_0}{s^2 + a_1 s + a_0}$$

is the same information as a linear state equation. In controllable canonical form, $\mathbf{x} = [x_1,\ x_2]$,

$$
\dot{\mathbf{x}} = f(\mathbf{x}, u, t) =  \begin{bmatrix} 0 & 1 \\ -a_0 & -a_1 \end{bmatrix} \mathbf{x} + \begin{bmatrix} 0 \\ 1 \end{bmatrix} u, \qquad y = h(\mathbf{x}, u, t) = \begin{bmatrix} b_0 & b_1 \end{bmatrix} \mathbf{x}.
$$




## Example — nonlinear mass–spring–damper

Parts 1 and 2 both simulate this plant. The damper force is proportional to $\dot x^3$:

$$m\ddot{x} = u - k x - b\dot{x}^3$$

The measured output is the whole state:

$$
\mathbf{y} = h(\mathbf{x}, u, t) = \mathbf{x}.
$$

The control law is proportional–derivative feedback on that output:

$$u = \mathrm{ctl}(\mathbf{y}, r, t) = k_p(r - x) - k_d\dot{x}.$$


## Part 1 — NumPy, SciPy and Matplotlib

Simulating means integrating $\dot{\mathbf{x}} = f(\mathbf{x}, u, t)$ over time: `solve_ivp` does it, given a Python function that returns $\dot{\mathbf{x}}$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

### Example — equations of motion

The mass–spring example, written as a first-order state equation. It does not read $t$:

$$\dot{\mathbf{x}} = f(\mathbf{x}, u, t) = \begin{bmatrix} \dot x \\ (u - k x - b \dot x^3)/m \end{bmatrix}, \qquad \mathbf{y} = h(\mathbf{x}, u, t) = \mathbf{x}.$$

In [ ]:
m, k, b = 1.0, 1.0, 1.0


def f(x, u, t):
    """ẋ = f(x, u, t): state x = [x, ẋ], input u = force (N), returns [ẋ, ẍ]."""
    pos, vel = x

    # m ẍ = u − k x − b ẋ³
    acc = (u - k * pos - b * vel**3) / m

    return np.array([vel, acc])

### Open loop

The input signal is a function of time, $u(t)$. `solve_ivp` expects a function $(t, \mathbf{x}) \mapsto \dot{\mathbf{x}}$: we write it by substituting the signal into the state equation, $\dot{\mathbf{x}} = f(\mathbf{x}, u(t), t)$.

In [ ]:
def input_signal(t):
    """u(t): a constant force of 1 N."""
    u = 1.0

    return u


def f_open_loop(t, x):
    """ẋ = f(x, u(t), t)"""
    u = input_signal(t)

    dx = f(x, u, t)

    return dx


x0 = np.array([0.0, 0.0])
t = np.linspace(0.0, 30.0, 3001)

sol = solve_ivp(f_open_loop, [0.0, 30.0], x0, t_eval=t)

plt.plot(sol.t, sol.y[0])
plt.xlabel("t [s]")
plt.ylabel("x [m]")
plt.grid(True)
plt.show()

### Closed loop

The reference is a function of time, $r(t)$, and the control law reads the output, the reference, and time, $u = \mathrm{ctl}(\mathbf{y}, r, t)$. Substituting into $f$ gives the closed-loop dynamics

$$\dot{\mathbf{x}} = f\bigl(\mathbf{x},\ \mathrm{ctl}(h(\mathbf{x}, u, t), r(t), t),\ t\bigr)$$

integrated the same way. In the example, $\mathbf{y} = \mathbf{x}$.

In [ ]:
kp, kd = 100.0, 5.0


def reference_signal(t):
    """r(t) """
    
    if t < 2.0:
        r = 0.0
    else:
        r = 1.0

    return r

def h(x,t):
    """y = h(x,t):."""

    # Note, usually the output is not a direct function of the input
    # but a function of the state and of time
    # if the output is a function of the input
    # then there are causality issues when we close the loop

    #  the output is the whole state for the example
    return x


def ctl(y, r, t):
    """u = ctl(y, r, t): state feedback, y = x = [x, ẋ]."""
    pos, vel = y

    u = kp * (r - pos) - kd * vel

    return u


def f_closed_loop(t, x):
    """ẋ = f(x, ctl(y, r(t), t), t), y = x"""
    y = h(x,t)
    r = reference_signal(t)
    u = ctl(y, r, t)

    dx = f(x, u, t)

    return dx


t = np.linspace(0.0, 10.0, 1001)

sol = solve_ivp(f_closed_loop, [0.0, 10.0], x0, t_eval=t)
u = [ctl(x, reference_signal(tk), tk) for tk, x in zip(sol.t, sol.y.T)]

fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(sol.t, sol.y[0])
ax[1].plot(sol.t, u)
ax[0].set_ylabel("x [m]")
ax[1].set_ylabel("u [N]")
ax[1].set_xlabel("t [s]")
ax[0].grid(True)
ax[1].grid(True)
plt.show()

## Part 2 — minilink (experimental toolbox!!)

[minilink](https://github.com/alx87grd/minilink) is a toolbox design to simplify doing those steps and create more complex diagram of interconnected dynamic system. Here we show so basic functionnality that reproduce the same steps done directly with scipy in part 1.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import numpy as np
from minilink import Controller, DynamicSystem, System

### Example — equations of motion

In minilink, a dynamic system is represented as an object with a `f` state equation $\dot{\mathbf{x}} = f(\mathbf{x}, u, t)$ and an `h` output equation $\mathbf{y} = h(\mathbf{x}, u, t)$. 

In [ ]:
m, k, b = 1.0, 1.0, 1.0


class NonlinearMassSpringDamper(DynamicSystem):

    def __init__(self):
        super().__init__(
            n=2,            # dimension of x
            input_dim=1,    # dimension of u
            output_dim=2,   # dimension of y
        )

    def f(self, x, u, t=0, params=None):
        pos, vel = x
        force = u[0]

        # m ẍ = u − k x − b ẋ³
        acc = (force - k * pos - b * vel**3) / m

        return np.array([vel, acc])

    def h(self, x, u, t=0, params=None):
        y = x

        return y


plant = NonlinearMassSpringDamper()

plant

### Open loop

The input signal is a source block: no state, one output $u(t)$. `>>` connects it to the plant.

In [ ]:
class InputSignal(System):
    """u(t): a constant force of 1 N."""

    def __init__(self):
        super().__init__()
        self.id = "source"
        self.add_output_port("u", function=self.signal)

    def signal(self, x, u, t=0, params=None):
        force = 1.0

        return force

source = InputSignal()

open_loop = source >> plant

open_loop.plot_diagram()

In [ ]:
open_loop.compute_trajectory(tf=30.0, verbose=False)

open_loop.plot_trajectory()

### Closed loop

The controller is a block with two inputs, the measurement $\mathbf{y}$ and the reference $r$, and one output $u = \mathrm{ctl}(\mathbf{y}, r, t)$. `ctl` receives the inputs concatenated in port order, and `dependencies="all"` declares that $u$ depends directly on them. `@` connects the plant output `y` to the controller input `y` and the controller output `u` to the plant input `u`: the port names do the wiring. The reference is another source block, connected with `>>`. 

In [ ]:
kp, kd = 100.0, 5.0


class MyCustomController(Controller):
    """u = ctl(y, r, t) = kp (r − x) − kd ẋ: inputs y = [x, ẋ] and r, output u = force (N)."""

    def __init__(self):
        super().__init__()
        self.add_input_port("y", dim=2)
        self.add_input_port("r")
        self.add_output_port("u", function=self.ctl, dependencies="all")

    def ctl(self, x, u, t=0, params=None):
        pos, vel, r = u  # the block inputs, in port order; t is available

        force = kp * (r - pos) - kd * vel

        return force


class ReferenceSignal(System):
    """r(t): a constant reference of 1 m."""

    def __init__(self):
        super().__init__()
        self.add_output_port("r", function=self.signal)

    def signal(self, x, u, t=0, params=None):

        if t < 2.0:
            r = 0.0
        else:
            r = 1.0

        return r

ctl = MyCustomController()
ref = ReferenceSignal()

closed_loop = ref >> ctl @ plant

closed_loop.plot_diagram()

In [ ]:
closed_loop.compute_trajectory(tf=10.0, verbose=False)

closed_loop.plot_trajectory()

### Bonus: the phase plane

The open-loop trajectory in the $(x, \dot x)$ plane, overlaid on the vector field $f(\mathbf x, u = 1)$.

In [ ]:
plant.plot_phase_plane(open_loop.traj, u=[1.0])

### Animating the simulation

If you define shapes and transformation placing those shape as a function of the plant state, minilkin can animate the resulting simulation:

The pose of the mass:

In [ ]:
def tf(self, x, u, t=0, params=None):
    """Place the mass at the position x: a pure translation."""
    pos = x[0]

    T = np.array(
        [
            [1.0, 0.0, 0.0, pos],
            [0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 1.0],
        ]
    )

    return {"body": T}

Defining shapes in either world frame or the mass body frame:

In [ ]:
from minilink.graphical.catalog import Arrow, Box, ground_line, spring_between


def get_kinematic_geometry(self):


    mass = Box(length_x=0.6, length_y=0.6, length_z=0.12, color="blue")

    return {"body": [mass]}


def get_dynamic_geometry(self, x, u, t=0, params=None):

    # Spring from a fixed wall to the mass
    spring = spring_between([-2.0, 0.0], [ x[0] - 0.3, 0.0])

    # Drawn length is scale * |vector|.
    force = float(u[0])
    direction = 1.0 if force >= 0.0 else -1.0
    arrow = Arrow(base=(0.35, 0.0), vector=(direction, 0.0), scale=0.3 * abs(force), color="red")

    return {"world": [spring], "body": [arrow]}


NonlinearMassSpringDamper.get_kinematic_geometry = get_kinematic_geometry
NonlinearMassSpringDamper.tf = tf
NonlinearMassSpringDamper.get_dynamic_geometry = get_dynamic_geometry

closed_loop.camera_scale = 3.0
closed_loop.animate()


In [ ]:
# open_loop.animate(renderer="meshcat")